## ⚙️ Configuration Kaggle — À faire avant tout

> Ce notebook a été **converti automatiquement** depuis Google Colab.

### Checklist de démarrage :
1. **Internet** : Panneau droite → *Settings* → **Internet: ON**
2. **GPU** : Panneau droite → *Settings* → **Accelerator: GPU T4 x2**
3. **API Roboflow** : Remplacez `VOTRE_CLE_ROBOFLOW` dans la cellule Roboflow
4. **Vidéo test** *(optionnel)* : *+ Add Data* → uploadez votre vidéo dashcam

### Différences vs Colab :
| Colab | Kaggle |
|-------|--------|
| `/content/` | `/kaggle/working/` |
| `files.download()` | Onglet **Output** à droite |
| `files.upload()` | Ajouter via **+ Add Data** |
| Config `kaggle.json` manuelle | **Automatique** dans Kaggle |


# 🚗 ADAS v6 — Notebook Complet Final
### LISA + GTSDB + DS1 + DS2 + DS3

In [1]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Fri May  1 04:48:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ultralytics roboflow kaggle -q
from ultralytics import YOLO
import os, cv2, yaml, random, shutil, pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
print("✓ Dépendances OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.1 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settin

In [3]:
import os
from pathlib import Path

for split in ['train', 'val']:
    Path(f'/kaggle/working/dataset/{split}/images').mkdir(parents=True, exist_ok=True)
    Path(f'/kaggle/working/dataset/{split}/labels').mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'vitesse_20', 'vitesse_30', 'vitesse_50', 'vitesse_60',
    'vitesse_70', 'vitesse_80', 'vitesse_100', 'vitesse_120',
    'depassement_interdit', 'stop', 'sens_interdit', 'entree_interdite',
    'feu_vert', 'feu_rouge', 'feu_orange'
]
print(f"✓ Structure créée | {len(CLASS_NAMES)} classes")

✓ Structure créée | 15 classes


In [4]:
import os
# ✅ Dans Kaggle, les credentials sont déjà disponibles automatiquement.
# Ajoutez votre clé API dans : Settings → Add-ons → API Token
print("✓ Kaggle API disponible nativement dans ce notebook")


✓ Kaggle API disponible nativement dans ce notebook


In [5]:
from roboflow import Roboflow
import yaml

API_KEY = "BJFyG6Ir6WuuW4vm3ceJ"  # ← remplacez ici
rf = Roboflow(api_key=API_KEY)
print("✓ Roboflow connecté")

✓ Roboflow connecté


In [6]:
# DS1 — Panneaux français
ds1 = rf.workspace("traffic-signs-detection-aojvk") \
        .project("traffic-signs-and-traffic-lights") \
        .version(1).download("yolov8", location="/kaggle/working/ds1")
with open(f"{ds1.location}/data.yaml") as f:
    ds1_yaml = yaml.safe_load(f)
print("✓ DS1:", len(ds1_yaml['names']), "classes")

# DS2 — Feux isolés
ds2 = rf.workspace("yolo-babua") \
        .project("traffic-light-detection-h8cvg") \
        .version(1).download("yolov8", location="/kaggle/working/ds2")
with open(f"{ds2.location}/data.yaml") as f:
    ds2_yaml = yaml.safe_load(f)
print("✓ DS2:", ds2_yaml['names'])

# DS3 — Panneaux anglais
ds3 = rf.workspace("usmanchaudhry622-gmail-com") \
        .project("traffic-and-road-signs") \
        .version(1).download("yolov8", location="/kaggle/working/ds3")
with open(f"{ds3.location}/data.yaml") as f:
    ds3_yaml = yaml.safe_load(f)
print("✓ DS3:", len(ds3_yaml['names']), "classes")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/ds1 in yolov8:: 100%|██████████| 8520/8520 [00:01<00:00, 7559.49it/s]

✓ DS1: 61 classes
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/ds2 in yolov8:: 100%|██████████| 18016/18016 [00:02<00:00, 7392.50it/s] 

✓ DS2: ['green', 'off', 'red', 'yellow']
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/ds3 in yolov8:: 100%|██████████| 20012/20012 [00:01<00:00, 10750.76it/s]

✓ DS3: 29 classes


In [7]:
import os

os.makedirs('/kaggle/working/gtsdb', exist_ok=True)

print("Téléchargement GTSDB Train (1GB)...")
!wget -q --show-progress "https://sid.erda.dk/public/archives/ff17dc924eba88d5d01a807357d6614c/TrainIJCNN2013.zip" -O /kaggle/working/gtsdb/Train.zip

print("Téléchargement GTSDB Test (530MB)...")
!wget -q --show-progress "https://sid.erda.dk/public/archives/ff17dc924eba88d5d01a807357d6614c/TestIJCNN2013.zip" -O /kaggle/working/gtsdb/Test.zip

print("Téléchargement gt.txt...")
!wget -q "https://sid.erda.dk/public/archives/ff17dc924eba88d5d01a807357d6614c/gt.txt" -O /kaggle/working/gtsdb/gt.txt

print("\nExtraction...")
import zipfile
with zipfile.ZipFile('/kaggle/working/gtsdb/Train.zip', 'r') as z:
    z.extractall('/kaggle/working/gtsdb/train')
with zipfile.ZipFile('/kaggle/working/gtsdb/Test.zip', 'r') as z:
    z.extractall('/kaggle/working/gtsdb/test')

from pathlib import Path
imgs = list(Path('/kaggle/working/gtsdb').rglob('*.ppm'))
print(f"✓ GTSDB prêt: {len(imgs)} images")

Téléchargement GTSDB Train (1GB)...
/kaggle/working/gts 100%[===================>]   1.03G  40.3MB/s    in 51s     
Téléchargement GTSDB Test (530MB)...
/kaggle/working/gts 100%[===================>] 529.93M  14.4MB/s    in 39s     
Téléchargement gt.txt...

Extraction...
✓ GTSDB prêt: 1753 images


In [8]:
import os

os.makedirs('/kaggle/working/lisa2', exist_ok=True)

print("Téléchargement LISA (4.5GB)...")
!kaggle datasets download -d mbornoe/lisa-traffic-light-dataset -p /kaggle/working/lisa2

print("Extraction LISA...")
!unzip -q /kaggle/working/lisa2/lisa-traffic-light-dataset.zip -d /kaggle/working/lisa2/data

from pathlib import Path
imgs = list(Path('/kaggle/working/lisa2/data').rglob('*.jpg'))
print(f"✓ LISA prêt: {len(imgs)} images")

Téléchargement LISA (4.5GB)...
Dataset URL: https://www.kaggle.com/datasets/mbornoe/lisa-traffic-light-dataset
License(s): CC-BY-NC-SA-4.0
100%|███████████████████████████████████████| 4.21G/4.21G [00:29<00:00, 152MB/s]

Extraction LISA...
✓ LISA prêt: 44075 images


In [9]:
import pandas as pd, cv2, random
from pathlib import Path

LISA_TO_ADAS = {
    'go': 12, 'goLeft': 12,
    'stop': 13, 'stopLeft': 13,
    'warning': 14,
}

print("Construction index LISA...")
img_index = {p.name: str(p) for p in Path('/kaggle/working/lisa2/data').rglob('*.jpg')}
print(f"Index: {len(img_index)} images")

csv_files = list(Path('/kaggle/working/lisa2/data').rglob('frameAnnotationsBOX.csv'))
all_annotations = {}

for csv_file in csv_files:
    df = pd.read_csv(csv_file, sep=';')
    for _, row in df.iterrows():
        tag = row['Annotation tag']
        if tag not in LISA_TO_ADAS: continue
        cls = LISA_TO_ADAS[tag]
        x1,y1,x2,y2 = int(row['Upper left corner X']), int(row['Upper left corner Y']), \
                       int(row['Lower right corner X']), int(row['Lower right corner Y'])
        img_name = Path(row['Filename']).name
        if img_name not in img_index: continue
        all_annotations.setdefault(img_index[img_name], []).append((x1,y1,x2,y2,cls))

print(f"Images annotées: {len(all_annotations)}")

imgs = list(all_annotations.keys())
random.shuffle(imgs)
split_idx = int(len(imgs) * 0.85)
converted = skipped = 0

for i, img_path in enumerate(imgs):
    img = cv2.imread(img_path)
    if img is None: skipped += 1; continue
    h, w = img.shape[:2]
    split  = 'train' if i < split_idx else 'val'
    fname  = Path(img_path).stem
    dst_img = f'/kaggle/working/dataset/{split}/images/{fname}_lisa.jpg'
    dst_lbl = f'/kaggle/working/dataset/{split}/labels/{fname}_lisa.txt'
    lines = []
    for (x1,y1,x2,y2,c) in all_annotations[img_path]:
        cx=((x1+x2)/2)/w; cy=((y1+y2)/2)/h
        bw=(x2-x1)/w;     bh=(y2-y1)/h
        lines.append(f"{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    cv2.imwrite(dst_img, img)
    Path(dst_lbl).write_text('\n'.join(lines))
    converted += 1

print(f"✓ LISA: {converted} images converties | {skipped} ignorées")

Construction index LISA...
Index: 43016 images
Images annotées: 36186
✓ LISA: 36186 images converties | 0 ignorées


In [10]:
import cv2, random
from pathlib import Path

GTSDB_TO_ADAS = {
    0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 7:6, 8:7,
    9:8, 10:8, 14:9, 15:10, 17:11
}

gt_file = '/kaggle/working/gtsdb/gt.txt'
img_dir = '/kaggle/working/gtsdb/train/TrainIJCNN2013'

annotations = {}
with open(gt_file) as f:
    for line in f:
        parts = line.strip().split(';')
        if len(parts) < 6: continue
        img_name = parts[0]
        x1,y1,x2,y2,cls = int(parts[1]),int(parts[2]),int(parts[3]),int(parts[4]),int(parts[5])
        if cls not in GTSDB_TO_ADAS: continue
        annotations.setdefault(img_name, []).append((x1,y1,x2,y2,GTSDB_TO_ADAS[cls]))

print(f"Annotations GTSDB: {len(annotations)} images")

imgs = list(annotations.keys())
random.shuffle(imgs)
split_idx = int(len(imgs) * 0.85)
converted = skipped = 0

for i, img_name in enumerate(imgs):
    img_path = f'{img_dir}/{img_name}'
    img = cv2.imread(img_path)
    if img is None: skipped += 1; continue
    h, w = img.shape[:2]
    split  = 'train' if i < split_idx else 'val'
    fname  = Path(img_name).stem
    dst_img = f'/kaggle/working/dataset/{split}/images/{fname}_gtsdb.jpg'
    dst_lbl = f'/kaggle/working/dataset/{split}/labels/{fname}_gtsdb.txt'
    lines = []
    for (x1,y1,x2,y2,c) in annotations[img_name]:
        cx=((x1+x2)/2)/w; cy=((y1+y2)/2)/h
        bw=(x2-x1)/w;     bh=(y2-y1)/h
        lines.append(f"{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    cv2.imwrite(dst_img, img)
    Path(dst_lbl).write_text('\n'.join(lines))
    converted += 1

print(f"✓ GTSDB: {converted} images converties | {skipped} ignorées")

Annotations GTSDB: 287 images
✓ GTSDB: 287 images converties | 0 ignorées


In [11]:
import shutil, os, random, yaml
from pathlib import Path

NAME_TO_ADAS = {
    # DS1 français
    'vitesse limitee a -20km-h-': 0, 'vitesse limitee a -30km-h-': 1,
    'vitesse limitee a -40km-h-': 1, 'vitesse limitee a -50km-h-': 2,
    'vitesse limitee a -60km-h-': 3, 'vitesse limitee a -70km-h-': 4,
    'vitesse limitee a -80km-h-': 5, 'vitesse limitee a -90km-h-': 5,
    'vitesse limitee a -100km-h-': 6, 'vitesse limitee a -130km-h-': 7,
    'vitesse limitee a -10km-h-': 0, 'vitesse limitee a -5km-h-': 0,
    'interdiction de depasser': 8, 'stop': 9, 'sens interdit': 10,
    'acces interdit aux vehicules depssant 5': 11,
    'acces interdit aux vehicules transportant de la marchandise': 11,
    # DS2 feux
    'green': 12, 'go': 12, 'red': 13, 'yellow': 14, 'off': -1,
    # DS3 anglais
    'speed limit 20 kmph': 0, 'speed limit 30 kmph': 1,
    '50 mph speed limit': 5, 'no entry': 10,
    'no_over_taking': 8, 'overtaking by trucks is prohibited': 8,
    'truck traffic is prohibited': 11, 'stop_sign': 9, 'traffic_signal': -1,
}

def normalize(name):
    return str(name).strip().lower().replace('_', ' ')

def remap_label_file(src_txt, dst_txt, src_names):
    lines_out = []
    try:
        content = Path(src_txt).read_text().strip()
        if not content: return 0
        for line in content.split('\n'):
            parts = line.strip().split()
            if not parts: continue
            src_cls = int(parts[0])
            if src_cls >= len(src_names): continue
            raw = normalize(src_names[src_cls])
            adas = NAME_TO_ADAS.get(raw)
            if adas is None or adas < 0: continue
            lines_out.append(f"{adas} {' '.join(parts[1:])}")
    except: return 0
    if not lines_out: return 0
    Path(dst_txt).parent.mkdir(parents=True, exist_ok=True)
    Path(dst_txt).write_text('\n'.join(lines_out))
    return len(lines_out)

def merge_dataset(ds_location, ds_yaml, tag, split_ratio=0.85):
    src_names = ds_yaml.get('names', [])
    total_imgs = total_boxes = skipped = 0
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = Path(ds_location) / split / 'images'
        lbl_dir = Path(ds_location) / split / 'labels'
        if not img_dir.exists(): continue
        imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        for img_path in imgs:
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists(): skipped += 1; continue
            dst_split = 'train' if random.random() < split_ratio else 'val'
            dst_img = f'/kaggle/working/dataset/{dst_split}/images/{img_path.stem}_{tag}.jpg'
            dst_lbl = f'/kaggle/working/dataset/{dst_split}/labels/{img_path.stem}_{tag}.txt'
            n = remap_label_file(str(lbl_path), dst_lbl, src_names)
            if n > 0:
                # ✅ Symlink au lieu de copie → 0 espace disque supplémentaire
                if not os.path.exists(dst_img):
                    os.symlink(str(img_path.resolve()), dst_img)
                total_imgs += 1; total_boxes += n
            else: skipped += 1
    print(f"  {tag}: {total_imgs} images | {total_boxes} bboxes | {skipped} ignorées")
    return total_imgs, total_boxes

# Vérification espace disque avant fusion
total, used, free = shutil.disk_usage('/kaggle/working')
print(f"💾 Disque — Total: {total/1e9:.1f}GB | Utilisé: {used/1e9:.1f}GB | Libre: {free/1e9:.1f}GB\n")

print("Fusion DS1...")
merge_dataset(ds1.location, ds1_yaml, 'ds1')
print("Fusion DS2...")
merge_dataset(ds2.location, ds2_yaml, 'ds2')
print("Fusion DS3...")
merge_dataset(ds3.location, ds3_yaml, 'ds3')
print("✓ Fusion terminée")

💾 Disque — Total: 21.0GB | Utilisé: 20.1GB | Libre: 0.8GB

Fusion DS1...
  ds1: 1751 images | 2071 bboxes | 2503 ignorées
Fusion DS2...
  ds2: 8921 images | 21984 bboxes | 81 ignorées
Fusion DS3...
  ds3: 2476 images | 2476 bboxes | 7524 ignorées
✓ Fusion terminée


In [12]:
import shutil, os
from pathlib import Path

bg_dirs = ['/kaggle/working/ds2/train/images', '/kaggle/working/lisa2/data']
added = 0
bg_imgs = []
for d in bg_dirs:
    if Path(d).exists():
        bg_imgs += list(Path(d).rglob('*.jpg'))[:200]

for i, p in enumerate(bg_imgs[:400]):
    dst = f'/kaggle/working/dataset/train/images/bg_{i:05d}.jpg'
    lbl = f'/kaggle/working/dataset/train/labels/bg_{i:05d}.txt'
    shutil.copy(str(p), dst)
    open(lbl, 'w').close()
    added += 1

print(f"✓ {added} background images ajoutées")

✓ 400 background images ajoutées


In [13]:
from pathlib import Path
from collections import Counter

CLASS_NAMES = [
    'vitesse_20','vitesse_30','vitesse_50','vitesse_60',
    'vitesse_70','vitesse_80','vitesse_100','vitesse_120',
    'depassement_interdit','stop','sens_interdit','entree_interdite',
    'feu_vert','feu_rouge','feu_orange'
]

print("="*55)
print("  AUDIT DU DATASET")
print("="*55)

for split in ['train', 'val']:
    imgs   = list(Path(f'/kaggle/working/dataset/{split}/images').glob('*.jpg'))
    labels = list(Path(f'/kaggle/working/dataset/{split}/labels').glob('*.txt'))
    counts = Counter()
    empty  = 0
    for lbl in labels:
        lines = lbl.read_text().strip().split('\n')
        if not lines or lines == ['']: empty += 1; continue
        for line in lines:
            parts = line.split()
            if parts: counts[int(parts[0])] += 1

    print(f"\n{split.upper()}: {len(imgs)} images | {empty} backgrounds")
    print(f"{'Classe':<30} {'Count':>6}")
    print("-"*40)
    max_c = max(counts.values()) if counts else 1
    for i, name in enumerate(CLASS_NAMES):
        c    = counts.get(i, 0)
        bar  = '█' * int(c/max_c*20)
        flag = ' ⚠️ MANQUANT' if c == 0 else ''
        print(f"  {i:2d}. {name:<27} {c:6d}  {bar}{flag}")

    feux = sum(counts.get(i,0) for i in [12,13,14])
    print(f"\n  Feux total: {feux}")
    if feux == 0:
        print("  🚨 CRITIQUE: Aucun feu!")

print("\n" + "="*55)

  AUDIT DU DATASET

TRAIN: 42586 images | 400 backgrounds
Classe                          Count
----------------------------------------
   0. vitesse_20                     342  
   1. vitesse_30                    1182  
   2. vitesse_50                     183  
   3. vitesse_60                     101  
   4. vitesse_70                     123  
   5. vitesse_80                     408  
   6. vitesse_100                     79  
   7. vitesse_120                     81  
   8. depassement_interdit           466  
   9. stop                           288  
  10. sens_interdit                  558  
  11. entree_interdite               418  
  12. feu_vert                     51189  █████████████████
  13. feu_rouge                    59914  ████████████████████
  14. feu_orange                    2596  

  Feux total: 113699

VAL: 7435 images | 0 backgrounds
Classe                          Count
----------------------------------------
   0. vitesse_20                      72  
   

In [14]:
import yaml

data = {
    'path': '/kaggle/working/dataset',
    'train': 'train/images',
    'val':   'val/images',
    'nc':    15,
    'names': CLASS_NAMES
}
with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)
print("✓ data.yaml créé")
print(open('/kaggle/working/dataset/data.yaml').read())

✓ data.yaml créé
names:
- vitesse_20
- vitesse_30
- vitesse_50
- vitesse_60
- vitesse_70
- vitesse_80
- vitesse_100
- vitesse_120
- depassement_interdit
- stop
- sens_interdit
- entree_interdite
- feu_vert
- feu_rouge
- feu_orange
nc: 15
path: /kaggle/working/dataset
train: train/images
val: val/images



In [15]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data      = '/kaggle/working/dataset/data.yaml',
    epochs    = 30,
    imgsz     = 640,
    batch     = 16,
    device    = 0,
    project   = '/kaggle/working/runs',
    name      = 'adas_v6',
    patience  = 25,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    degrees=5.0, translate=0.1, scale=0.5,
    fliplr=0.0, mosaic=1.0, mixup=0.15,
    optimizer='AdamW', lr0=0.001, lrf=0.01,
    weight_decay=0.0005,
    save=True, save_period=10, plots=True, verbose=True,
)
print("\n✅ Entraînement terminé!")

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=adas_v6, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patienc

In [16]:
from ultralytics import YOLO

best = YOLO('/kaggle/working/runs/adas_v6/weights/best.pt')
m = best.val(data='/kaggle/working/dataset/data.yaml', imgsz=640)

print(f"mAP50    : {m.box.map50:.3f}")
print(f"mAP50-95 : {m.box.map:.3f}")
print(f"Précision: {m.box.mp:.3f}")
print(f"Rappel   : {m.box.mr:.3f}")
print()
for i, name in enumerate(CLASS_NAMES):
    ap   = m.box.ap50[i] if i < len(m.box.ap50) else 0
    flag = '✅' if ap >= 0.5 else ('⚠️' if ap >= 0.2 else '❌')
    print(f"  {flag} {name:<30} AP50={ap:.3f}")

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,131,389 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1769.9±954.9 MB/s, size: 125.0 KB)
val: Scanning /kaggle/working/dataset/val/labels.cache... 7435 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7435/7435 2.2Git/s 0.0s
val: /kaggle/working/dataset/val/images/dayClip6--00006_lisa.jpg: 3 duplicate labels removed
val: /kaggle/working/dataset/val/images/dayClip6--00007_lisa.jpg: 2 duplicate labels removed
val: /kaggle/working/dataset/val/images/dayClip6--00012_lisa.jpg: 3 duplicate labels removed
val: /kaggle/working/dataset/val/images/dayClip6--00021_lisa.jpg: 3 duplicate labels removed
val: /kaggle/working/dataset/val/images/dayClip6--00026_lisa.jpg: 3 duplicate labels removed
val: /kaggle/working/dataset/val/images/dayClip6--00036_lisa.jpg: 4 duplicate labels removed
val: /kaggle/working/dataset/val/images/dayClip6--000

In [17]:
# Test sur vidéo dashcam
# ➡️  Ajoutez votre vidéo via "+ Add Data" puis décommentez :
# video_path = "/kaggle/input/NOM_DATASET/votre_video.mp4"
# best.predict(
#     source  = video_path,
#     conf    = 0.45,
#     iou     = 0.5,
#     save    = True,
#     project = '/kaggle/working/predictions',
#     name    = 'test_v6',
# )
print("→ Ajoutez votre vidéo via '+ Add Data' puis décommentez le bloc ci-dessus")


→ Ajoutez votre vidéo via '+ Add Data' puis décommentez le bloc ci-dessus


In [18]:
import shutil

shutil.copy('/kaggle/working/runs/adas_v6/weights/best.pt', '/kaggle/working/best_adas_v6.pt')
# ✅ Fichier dispo dans l'onglet "Output" à droite → bouton Download
print("✅ Modèle téléchargé — remplacez models/best.pt dans votre projet ADAS")

✅ Modèle téléchargé — remplacez models/best.pt dans votre projet ADAS


In [19]:
import glob, shutil

# Sauvegarde automatique dès la fin
best = glob.glob('/kaggle/working/**/best.pt', recursive=True)[0]
shutil.copy(best, '/kaggle/working/best_adas_v6.pt')
print(f"✓ Modèle copié depuis : {best}")

# Télécharge IMMÉDIATEMENT depuis l'onglet Output →

✓ Modèle copié depuis : /kaggle/working/runs/adas_v6/weights/best.pt
